# 1 · Prepare notes for annotation (MedCAT + MedCATtrainer)

This notebook turns **any set of clinical notes** into an annotation round for MedCATtrainer:

1. Load the notes (CSV or XLSX) and split them into an **annotation (train) set** and a **test set**.
2. Build the concept database (CDB) from a SNOMED CT concept list, or continue from an earlier model pack.
3. Give the vocabulary word vectors from **`CLTL/MedRoBERTa.nl`**, and run unsupervised MedCAT training on the notes, so MedCATtrainer suggests sensible spans.
4. Save a **model pack for MedCATtrainer** (CDB + Vocab, no MetaCAT) and the two **dataset CSVs** to upload.

Next: annotate both sets in MedCATtrainer (instructions at the end), then train with `02_medcat_train_medroberta.ipynb` and evaluate with `03_medcat_evaluate.ipynb`.

## Configuration
Edit these settings, then run all cells.

In [ ]:
from pathlib import Path

# --- Input notes -------------------------------------------------------------
NOTES_PATH = "medcat_data/notes/my_notes.csv"   # CSV or XLSX, one note per row (de-identified!)
TEXT_COLUMN = "text"                            # column with the note text
NAME_COLUMN = "name"                            # unique document id; generated if the column is missing
TEST_FRACTION = 0.2                             # share of notes held out as the test set (notebook 3); 0 = none
RANDOM_SEED = 42

# --- Model -------------------------------------------------------------------
ROUND_TAG = "round1"          # names the output files of this annotation round
OVERWRITE_EXISTING = False    # True: replace an existing model pack with the same ROUND_TAG
MODEL_NAME = "snomed_symptoms_en_nl"
CONCEPTS_CSV = None           # MedCAT concept CSV (cui,name,ontologies,name_status,type_ids); None = starter symptom list
START_FROM_MODEL_PACK = None  # e.g. "medcat_data/models/snomed_symptoms_en_nl_annotation_v1.zip" from notebook 2
                              # (the MetaCAT-free pack) to prepare a next round with an already trained model
UNSUPERVISED_EPOCHS = 1
VOCAB_MIN_COUNT = 1           # words seen fewer times in the notes get no vector
VOCAB_MAX_WORDS = 50_000      # most frequent words to embed with MedRoBERTa.nl

MEDROBERTA_MODEL = "CLTL/MedRoBERTa.nl"
DATA_DIR = Path("medcat_data")
MODELS_DIR = DATA_DIR / "models"
ANNOTATION_DIR = DATA_DIR / "annotation_sets"
for d in (MODELS_DIR, ANNOTATION_DIR):
    d.mkdir(parents=True, exist_ok=True)

def free_pack_name(name):
    """Model pack folder + zip for `name`; MedCAT won't overwrite an existing pack."""
    folder = MODELS_DIR / name
    zip_path = folder.with_suffix(".zip")
    if folder.exists() or zip_path.exists():
        if not OVERWRITE_EXISTING:
            raise FileExistsError(f"{zip_path} already exists. Change the tag in the configuration, "
                                  "or set OVERWRITE_EXISTING = True to replace it.")
        import shutil
        shutil.rmtree(folder, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
    return name

## Load and split the notes
If `NOTES_PATH` doesn't exist yet, a small demo set is written there so the notebook runs end to end. Replace it with your own notes.

In [ ]:
import pandas as pd

DEMO_NOTES = [
    "Patiënt klaagt sinds twee dagen over kortademigheid en prikkelhoest. Geen koorts gemeten.",
    "Anamnese: vermoeidheid en benauwdheid bij traplopen. Ontkent pijn op de borst.",
    "Patient presents with acute chest pain and dyspnea on exertion. No fever or cough.",
    "Controle poli longziekten: dyspneu stabiel, geen verhoging, moeheid neemt iets af.",
    "Patiënt presenteert zich op de SEH met pijn op de borst en kortademigheid. Geen koorts.",
    "Bekend met COPD, nu toename van hoesten en vermoeidheid zonder benauwdheid.",
    "64yo male with chest pain radiating to left arm and dyspnea. Denies fever.",
    "Sinds een week koorts en hoesten, geen thoracale pijn.",
    "Patient reports fatigue and shortness of breath, no cough.",
    "Geen klachten van kortademigheid of vermoeidheid, wel prikkelhoest.",
]

notes_path = Path(NOTES_PATH)
if not notes_path.exists():
    print(f"{notes_path} not found: writing a demo set there. Replace it with your own de-identified notes.")
    notes_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"name": [f"demo_{i:03d}" for i in range(len(DEMO_NOTES))], "text": DEMO_NOTES}).to_csv(notes_path, index=False)

if notes_path.suffix.lower() in (".xlsx", ".xls"):
    raw = pd.read_excel(notes_path)  # needs: pip install openpyxl
else:
    raw = pd.read_csv(notes_path)
if TEXT_COLUMN not in raw.columns:
    raise ValueError(f"Column {TEXT_COLUMN!r} not in {list(raw.columns)}")

notes = pd.DataFrame({"text": raw[TEXT_COLUMN].fillna("").astype(str).str.strip()})
if NAME_COLUMN in raw.columns:
    notes["name"] = raw[NAME_COLUMN].astype(str)
else:
    notes["name"] = [f"doc_{i:06d}" for i in range(len(raw))]
notes = notes[notes["text"].str.len() > 0]
# MedCATtrainer needs unique document names
dup = notes["name"].duplicated(keep=False)
notes.loc[dup, "name"] = notes.loc[dup, "name"] + "_" + notes[dup].groupby("name").cumcount().astype(str)
notes = notes[["name", "text"]].reset_index(drop=True)

test_notes = notes.sample(frac=TEST_FRACTION, random_state=RANDOM_SEED) if TEST_FRACTION > 0 else notes.iloc[0:0]
train_notes = notes.drop(test_notes.index)
print(f"{len(notes)} notes: {len(train_notes)} for annotation/training, {len(test_notes)} for testing")

## Concept database (CDB)
Without `CONCEPTS_CSV`, a starter list of 5 SNOMED CT symptom concepts with English and Dutch names is used. Once your Nictiz SNOMED CT licence is in place, export the concepts you need to a CSV with the columns `cui,name,ontologies,name_status,type_ids` (`name_status`: `P` = preferred term, `A` = synonym) and set `CONCEPTS_CSV`.

In [ ]:
from medcat.cat import CAT
from medcat.config import Config
from medcat.model_creation.cdb_maker import CDBMaker
from medcat.vocab import Vocab

STARTER_CONCEPTS = [
    # (SNOMED CT id, name, name_status)
    ("267036007", "Dyspnea", "P"), ("267036007", "Shortness of breath", "A"), ("267036007", "Kortademigheid", "P"),
    ("267036007", "Benauwdheid", "A"), ("267036007", "Dyspneu", "A"),
    ("29857009", "Chest pain", "P"), ("29857009", "Pijn op de borst", "P"), ("29857009", "Thoracale pijn", "A"),
    ("386661006", "Fever", "P"), ("386661006", "Pyrexia", "A"), ("386661006", "Koorts", "P"), ("386661006", "Verhoging", "A"),
    ("84229001", "Fatigue", "P"), ("84229001", "Vermoeidheid", "P"), ("84229001", "Moeheid", "A"),
    ("49727002", "Cough", "P"), ("49727002", "Hoesten", "P"), ("49727002", "Prikkelhoest", "A"),
]

if START_FROM_MODEL_PACK:
    cat = CAT.load_model_pack(START_FROM_MODEL_PACK)
    if cat.get_addons():
        raise ValueError("Use the MetaCAT-free '..._annotation_<tag>.zip' pack from notebook 2, not the MedRoBERTa pack.")
    concept_names = [n.replace(cat.config.general.separator, " ") for n in cat.cdb.name2info]
else:
    concepts_csv = CONCEPTS_CSV
    if concepts_csv is None:
        concepts_csv = DATA_DIR / "starter_snomed_symptoms_en_nl.csv"
        pd.DataFrame([{"cui": c, "name": n, "ontologies": "SNOMED-CT", "name_status": s, "type_ids": "T184"}
                      for c, n, s in STARTER_CONCEPTS]).to_csv(concepts_csv, index=False)
    config = Config()
    config.components.ner.min_name_len = 3
    config.components.linking.similarity_threshold = 0.25
    cdb = CDBMaker(config).prepare_csvs([str(concepts_csv)], full_build=True)
    cat = CAT(cdb=cdb, vocab=Vocab(), config=config)
    concept_names = pd.read_csv(concepts_csv)["name"].astype(str).tolist()

print(f"CDB: {len(cat.cdb.cui2info)} concepts, {len(cat.cdb.name2info)} names")

## Vocabulary vectors from MedRoBERTa.nl
MedCAT disambiguates concepts with context vectors built from the word vectors around a mention. Each word in the notes gets a vector: the mean of MedRoBERTa.nl's token embeddings for that word. Words that already have a vector (when continuing from a model pack) are kept.

In [ ]:
from collections import Counter
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
hf_tokenizer = AutoTokenizer.from_pretrained(MEDROBERTA_MODEL)
hf_model = AutoModel.from_pretrained(MEDROBERTA_MODEL).to(device).eval()


@torch.no_grad()
def embed_words(words, batch_size=128):
    """Mean-pooled, L2-normalised MedRoBERTa.nl embedding per word."""
    out = []
    for i in range(0, len(words), batch_size):
        enc = hf_tokenizer(words[i:i + batch_size], return_tensors="pt", padding=True,
                           truncation=True, max_length=16).to(device)
        hidden = hf_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        vecs = torch.nn.functional.normalize((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9), dim=1)
        out.append(vecs.cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, hf_model.config.hidden_size))


word_counts = Counter(
    tkn.base.lower
    for text in notes["text"].tolist() + concept_names
    for tkn in cat.pipe.tokenizer(text)
    if tkn.base.lower.strip()
)
new_words = [w for w, c in word_counts.most_common(VOCAB_MAX_WORDS)
             if c >= VOCAB_MIN_COUNT and (w not in cat.vocab or cat.vocab.vec(w) is None)]
print(f"{len(word_counts)} distinct words, embedding {len(new_words)} with {MEDROBERTA_MODEL} on {device}...")
for word, vec in zip(new_words, embed_words(new_words)):
    cat.vocab.add_word(word, cnt=word_counts[word], vec=vec, replace=True)
cat.vocab.init_cumsums()

# Free memory before training
del hf_model
if device == "cuda":
    torch.cuda.empty_cache()

## Unsupervised training
Learns context vectors for the concepts from the annotation notes (not the test notes, to keep the test set unseen). Only mentions with a single possible concept are used, so no labels are needed.

In [ ]:
cat.trainer.train_unsupervised(train_notes["text"].tolist(), nepochs=UNSUPERVISED_EPOCHS)

## Save the MedCATtrainer model pack and datasets

In [ ]:
pack_folder = cat.save_model_pack(str(MODELS_DIR), pack_name=free_pack_name(f"{MODEL_NAME}_annotation_{ROUND_TAG}"),
                                  add_hash_to_pack_name=False)
annotation_pack = pack_folder + ".zip"

train_csv = ANNOTATION_DIR / f"{ROUND_TAG}_train.csv"
test_csv = ANNOTATION_DIR / f"{ROUND_TAG}_test.csv"
train_notes.to_csv(train_csv, index=False)
if len(test_notes):
    test_notes.to_csv(test_csv, index=False)

print("Upload to MedCATtrainer:")
print("  model pack:          ", annotation_pack)
print("  annotation dataset:  ", train_csv)
if len(test_notes):
    print("  test dataset:        ", test_csv)

## Preview: what MedCATtrainer will suggest

In [ ]:
for text in train_notes["text"].head(5):
    ents = cat.get_entities(text)["entities"].values()
    print("\n" + text[:200])
    for e in ents:
        print(f"  -> {e['source_value']!r}  {e['cui']} ({e['pretty_name']})  acc={e['acc']:.2f}")

## Annotate in MedCATtrainer

MedCATtrainer must be **v3.x** (MedCAT 2); see `README.md` for starting it. In the admin site (http://localhost:8001/admin/):

1. **Model packs → Add:** upload the model pack printed above (`..._annotation_<round>.zip`).
2. **Meta tasks → Add** (once): `Status` with values `Affirmed` and `Negated`. Notebook 2 expects exactly these names.
3. **Datasets → Add:** upload `<round>_train.csv`, and separately `<round>_test.csv`.
4. **Project annotate entities → Add:** create **two projects** with the same model pack and the `Status` task:
   - `<round> train` on the train dataset. *Train Model On Submit* is optional (better suggestions while annotating).
   - `<round> test` on the test dataset. **Untick** *Train Model On Submit*, so the test annotations don't feed back into the model.
5. Annotate both projects at http://localhost:8001/ (see `README.md` for the annotation steps and guidelines).
6. **Project annotate entities:** tick a project → *Download annotations* (with text) → *Go*. Save the exports as:
   - `medcat_data/exports/<round>_train_export.json` → notebook 2
   - `medcat_data/exports/<round>_test_export.json` → notebook 3